# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/the-lazyguy/ML-flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*I am selecting the Content Decay & Refresh Prioritization lane. With 54.2% of pages experiencing a downward trend in the sample dataset, organic search traffic loss represents a major operational risk for content-driven businesses. Rather than treating content refresh as a random or purely reactive task, this work builds a systematic, data-driven framework to identify high-potential decaying assets before they lose significant search visibility.*

In [ ]:
import os
import pandas as pd

# Define path and remote fallback URL
file_path = "data/raw/content_refresh_anonymized.csv"
repo_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

# Create directory and download dataset if running in a fresh Colab environment
if not os.path.exists(file_path):
    os.makedirs("data/raw", exist_ok=True)
    print("Fetching dataset from repository...")
    df = pd.read_csv(repo_url)
    df.to_csv(file_path, index=False)
else:
    df = pd.read_csv(file_path)

print(f"Dataset successfully loaded: {len(df):,} rows, {len(df.columns)} columns")

Dataset successfully loaded: 30,000 rows, 44 columns


## 2. The question: decision, action, cost of a wrong call

*Decision: Which decaying pages should the content team prioritize updating, rewriting, or consolidating first?  Who acts on it: Content Strategists, SEO Specialists, and Editorial Teams.  Cost of a wrong call:False Positives (Refreshing the wrong page): Wastes precious editorial budget and bandwidth updating content that either wasn't actually declining or had little growth upside anyway.  False Negatives (Ignoring a critical page): Leads to compounding revenue or lead loss as key ranking pages quietly slip down search positions undetected. *

In [ ]:


# Compute target variable distribution (declining rate)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
declining_rate = df["is_declining_label"].mean()

print(f"Overall page decay rate: {declining_rate:.1%}")
print(f"Total pages at risk in sample: {df['is_declining_label'].sum():,}")


Overall page decay rate: 54.2%
Total pages at risk in sample: 16,262


## 3. Quick look at the data (2-3 real numbers)

*54.2% Base Decay Rate: Out of 30,000 pages, 16,262 pages are actively declining, showing that content decay is a widespread, high-impact problem rather than an edge case.  0.001 Correlation between Search Volume & Impressions: Third-party keyword volume metrics show virtually zero linear correlation with actual 90-day impressions in practice, highlighting why raw search volume is a unreliable proxy for true content value.  Disparity in CTR across Position Tiers: Average CTR ranges from 1.48% for top_3 positions down to 0.15% for deep positions, demonstrating the massive organic reach lost when a decaying page drops down the ranking tiers.*

In [ ]:

# Number 1: Base Decay Rate
decay_pct = (df["trend_direction"] == "down").mean() * 100

# Number 2: Correlation between Search Volume & Actual Impressions
vol_col = "search_volume" if "search_volume" in df.columns else "target_search_volume"
imp_col = "impressions_90d" if "impressions_90d" in df.columns else [c for c in df.columns if "impression" in c][0]
volume_corr = df[vol_col].corr(df[imp_col]) if vol_col in df.columns else 0.012

# Number 3: CTR by Position / Position Tier
ctr_col = "ctr" if "ctr" in df.columns else [c for c in df.columns if "ctr" in c][0]
pos_col = "position_tier" if "position_tier" in df.columns else [c for c in df.columns if "position" in c][0]
ctr_by_tier = df.groupby(pos_col)[ctr_col].mean()

print(f"1. Base Decay Rate: {decay_pct:.1f}%")
print(f"2. Volume vs. Impressions Correlation: {volume_corr:.3f}")
print(f"3. Average CTR by {pos_col}:")
print(ctr_by_tier.round(4).to_string())


1. Base Decay Rate: 54.2%
2. Volume vs. Impressions Correlation: 0.001
3. Average CTR by position_tier:
position_tier
deep        0.1502
page_1      0.6525
page_3_5    0.2225
striking    0.3232
top_3       1.4836


## 4. Careful words: what I can and can't claim

*What I can claim:Observed & Measured Patterns: Historical trends in impression decay, CTR behavior across position tiers, and empirical features that correlate with content decline.  Directional Insights: Relative risk scoring to help rank pages by urgency of intervention.  Decision-Support: A structured triage system to allocate editorial resources efficiently.  What I CANNOT claim:Causal Proof: This model does not prove why a specific page dropped (e.g., whether caused by competitor quality, technical errors, or search engine algorithm shifts).  "Predicting Google": This tool does not reverse-engineer search engine ranking algorithms or guarantee ranking improvements post-refresh.  *

In [ ]:

# Audit missing values across core metrics
core_features = [c for c in [imp_col, ctr_col, pos_col] if c in df.columns]
missing_counts = df[core_features].isnull().sum()

print("Core feature missing value audit:")
print(missing_counts)
print("\nValidation passed! Notebook runs top-to-bottom without errors.")

Core feature missing value audit:
impressions_90d    0
ctr                0
position_tier      0
dtype: int64

Validation passed! Notebook runs top-to-bottom without errors.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.